# Approximate Inference using Gibbs Sampling

## 1. Imports and Model Setup

In [40]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from pathlib import Path
from pgmpy.inference import VariableElimination
from parameter_inference import load_fitted_model, exact_popularity

In [41]:
SEED = 150

with open("models/parameter_config.json") as f:
    config = json.load(f)

model = load_fitted_model(config["model_path"])

print("Structure:", config["structure"])
print("Estimator:", config["estimator"])
print("Equivalent Sample Size:", config["equivalent_sample_size"])
print("Valid Model:", model.check_model())

Structure: HC start=hand
Estimator: BDeu
Equivalent Sample Size: 20.0
Valid Model: True


In [42]:
queries = config["query_specs"]
popularity_states = config["state_names"]["popularity"]

ve_results = pd.read_csv("models/ve_queries.csv")

ve_results = ve_results[
    ve_results["model_id"] == config["primary_model_id"]
].copy()

print("Number of Queries:", len(queries))
print("Popularity States:", popularity_states)

ve_results.head()

Number of Queries: 80
Popularity States: ['(-0.001, 20.0]', '(20.0, 40.0]', '(40.0, 60.0]', '(60.0, 80.0]', '(80.0, 100.0]']


,model_id,query_id,experiment,evidence_json,train_evidence_count,popularity_state,probability
400,HC start=hand,prior,prior,{},3571,"(-0.001, 20.0]",0.069618
401,HC start=hand,prior,prior,{},3571,"(20.0, 40.0]",0.192982
402,HC start=hand,prior,prior,{},3571,"(40.0, 60.0]",0.343358
403,HC start=hand,prior,prior,{},3571,"(60.0, 80.0]",0.335283
404,HC start=hand,prior,prior,{},3571,"(80.0, 100.0]",0.058758


## 2. Markov Blanket Conditional Probabilities

In [43]:
print("Variables:", list(model.nodes()))
print("Parents of popularity:", model.get_parents("popularity"))
print("Children of popularity:", model.get_children("popularity"))
print("Markov blanket:", model.get_markov_blanket("popularity"))

Variables: ['mode', 'key', 'popularity', 'energy', 'tempo', 'valence', 'danceability', 'acousticness', 'decade', 'duration']
Parents of popularity: ['decade']
Children of popularity: ['acousticness']
Markov blanket: ['decade', 'acousticness']


In [44]:
state_names = {}

for node in model.nodes():
    state_names[node] = model.get_cpds(node).state_names[node]

for node, states in state_names.items():
    print(node, ":", states)

mode : ['0.0', '1.0']
key : ['0.0', '1.0', '10.0', '11.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0']
popularity : ['(-0.001, 20.0]', '(20.0, 40.0]', '(40.0, 60.0]', '(60.0, 80.0]', '(80.0, 100.0]']
energy : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
tempo : ['fast', 'mid', 'slow', 'upbeat']
valence : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
danceability : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
acousticness : ['(-0.001, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 1.0]']
decade : ['1990s', '2000s', '2010s', '2020s', 'Before 1990']
duration : ['2-3 min', '3-4 min', '4-5 min', '5+ min', '< 2 min']


In [45]:
cpd = model.get_cpds("popularity")

assignment = {}

for node in cpd.variables:
    assignment[node] = state_names[node][0]

print("Assignment:", assignment)
print("Probability:", cpd.get_value(**assignment))

Assignment: {'popularity': '(-0.001, 20.0]', 'decade': '1990s'}
Probability: 0.06037735849056604


In [46]:
def get_cpt_probability(model, node, assignment):
    cpd = model.get_cpds(node)

    values = {
        var: assignment[var]
        for var in cpd.variables
    }

    return cpd.get_value(**values)

In [47]:
def markov_blanket_distribution(model, node, assignment):
    probabilities = []

    for state in state_names[node]:
        current = assignment.copy()
        current[node] = state

        probability = get_cpt_probability(model, node, current)

        for child in model.get_children(node):
            # Multiply probability by the child's CPT probability
            # TODO

            probabilities.append(probability)

    probabilities = np.array(probabilities, dtype=float)

    return probabilities / probabilities.sum()

In [48]:
assignment = {
    node: states[0]
    for node, states in state_names.items()
}

probs = markov_blanket_distribution(
    model, "popularity", assignment
)

for state, prob in zip(popularity_states, probs):
    print(state, ":", prob)

print("Total:", probs.sum())

(-0.001, 20.0] : 0.06037735849056604
(20.0, 40.0] : 0.13584905660377358
(40.0, 60.0] : 0.17830188679245282
(60.0, 80.0] : 0.560377358490566
(80.0, 100.0] : 0.06509433962264151
Total: 1.0
